## Ingest Dimension Data into Bronze Layer

In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [ ]:
catalog_name = "ecommerce"
raw_data_path = '/Volumes/ecommerce/source_data/raw'

### Brand Dataframe & Table

In [ ]:
# Define schema struture..
brand_schema = StructType([
    StructField('brand_code', StringType()),
    StructField('brand_name', StringType()),
    StructField('category_code', StringType())
])

# prepared dataframe..
brand_df = spark.read.option("header", True).option("delimeter", ",").schema(brand_schema).csv(f"{raw_data_path}/brands/brands.csv")

# add some extra columns..
brand_df = brand_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(brand_df.limit(2))

# convert df to table.
brand_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_brands")

In [ ]:
%%sql
SELECT * FROM ecommerce.broze.brz_brands LIMIT 2;

### Category Dataframe & Table

In [ ]:
# Define schema struture..
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# prepared dataframe..
category_df = spark.read.option("header", True).option("delimeter", ",").schema(category_schema).csv(f"{raw_data_path}/category/category.csv")

# add some extra columns..
category_df = category_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(category_df.limit(2))

# convert df to table.
category_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_categories")

In [ ]:
%%sql
SELECT * FROM ecommerce.broze.brz_categories LIMIT 2;

### Product Dataframe & Table

In [ ]:
# Define schema struture..
product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# prepared dataframe..
product_df = spark.read.option("header", True).option("delimeter", ",").schema(product_schema).csv(f"{raw_data_path}/products/products.csv")

# add some extra columns..
product_df = product_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(product_df.limit(2))

# convert df to table.
product_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_products")

In [ ]:
%%sql
SELECT * FROM ecommerce.broze.brz_products LIMIT 2;

### Customer Dataframe & Table

In [ ]:
# Define schema struture..
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# prepared dataframe..
customer_df = spark.read.option("header", True).option("delimeter", ",").schema(customer_schema).csv(f"{raw_data_path}/customers/customers.csv")

# add some extra columns..
customer_df = customer_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(customer_df.limit(2))

# convert df to table.
customer_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_customers")

In [ ]:
%%sql
SELECT * FROM ecommerce.broze.brz_customers LIMIT 2;

### Date Dataframe & Table

In [ ]:
# Define schema struture..
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# prepared dataframe..
date_df = spark.read.option("header", True).option("delimeter", ",").schema(date_schema).csv(f"{raw_data_path}/date/date.csv")

# add some extra columns..
date_df = date_df.withColumn("created_at", F.current_timestamp()) \
                    .withColumn("source_file", F.col("_metadata.file_path"))

display(date_df.limit(2))

# convert df to table.
date_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog_name}.broze.brz_date")

In [ ]:
%%sql
SELECT * FROM ecommerce.broze.brz_date LIMIT 2;

In [ ]:
%%sql
use catalog `ecommerce`; select * from `broze`.`brz_order_items` limit 100;